# Lookup-table check on the chip

Plots one `test_lut_range()` run from `sw/tests/vco_test.py` (`results/vco/vco_lut_check_*.csv`). For each target frequency, `configure_vco` picks a setting from the lookup table. The test then sweeps Vctrl over the table's points, plus the predicted operating Vctrl and ±25 mV around it, and compares each measurement with the table.

- **Per target:** the table curve, the measured points, and the operating point.
- **Deviation:** measured vs table, in %, for all points.
- **Kvco:** measured local |Kvco| vs predicted.

Set `CHECK_CSV` below (`None` = newest run) and run all cells.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

VCO_DIR = Path.cwd().resolve().parents[2] / "results" / "vco"  # notebook lives in sw/tools/notebooks/
CHECK_CSV = None  # e.g. VCO_DIR / "vco_lut_check_S5_20261002_120000.csv"; None = newest test_lut_range run
F_TOL, KVCO_TOL = 0.03, 0.25  # test_lut_pick defaults

if CHECK_CSV is None:  # newest file with the test_lut_range columns (single test_lut_pick runs before it lack f_target)
    found = [p for p in sorted(VCO_DIR.glob("vco_lut_check_*.csv"), key=lambda p: p.stat().st_mtime)
             if "f_target" in pd.read_csv(p, nrows=0).columns]
    CHECK_CSV = found[-1]
df = pd.read_csv(CHECK_CSV)
points = df[df["kind"].isin(["table", "operating", "kvco"])]
summary = df[df["kind"].isin(["summary", "unreachable"])].sort_values("f_target")
print(f"{Path(CHECK_CSV).name}: table {df['lut_csv'].dropna().iloc[0]}, {summary['f_target'].nunique()} targets, "
      f"{len(points)} measured points, {(summary['result'] == 'PASS').sum()} PASS")

summary_table = pd.DataFrame({
    "f_target [MHz]": summary["f_target"] / 1e6,
    "setting": [f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}"
                if r.kind == "summary" else "-" for r in summary.itertuples()],
    "Vctrl pred [V]": summary["vctrl_pred"],
    "f at Vctrl pred [MHz]": summary["f_vco"] / 1e6,
    "f error [%]": (summary["f_vco"] / summary["f_target"] - 1) * 100,
    "worst dev [%]": summary["dev_pct"],
    "|Kvco| pred [MHz/V]": summary["kvco_pred"] / 1e6,
    "|Kvco| meas [MHz/V]": summary["kvco_meas"] / 1e6,
    "result": summary["result"],
}).reset_index(drop=True)
summary_table.round(3)

vco_lut_check_S5_20261002_102659.csv: table vco_lut_S5_20261001_161638.csv, 13 targets, 180 measured points, 10 PASS


,f_target [MHz],setting,Vctrl pred [V],f at Vctrl pred [MHz],f error [%],worst dev [%],|Kvco| pred [MHz/V],|Kvco| meas [MHz/V],result
0,50.0,-,NaN,NaN,NaN,NaN,NaN,NaN,UNREACHABLE
1,100.0,c15 min 15 max 15,0.620,95.107,-4.893,4.854,981.994,1082.265,FAIL
2,200.0,c15 min 15 max 15,0.545,201.634,0.817,2.145,942.202,1228.923,FAIL
3,300.0,c12 min 15 max 15,0.391,300.040,0.013,0.628,90.149,87.365,PASS
4,500.0,c12 min 12 max 15,0.376,499.588,-0.082,0.249,39.131,44.930,PASS
5,700.0,c14 min 15 max 12,0.373,699.941,-0.008,1.616,21.234,25.566,PASS
6,1000.0,c10 min 12 max 8,0.372,999.959,-0.004,0.380,23.772,24.064,PASS
7,1500.0,c5 min 14 max 12,0.379,1508.254,0.550,0.992,261.064,309.658,PASS
8,2000.0,c2 min 9 max 14,0.378,2000.561,0.028,0.265,297.807,284.644,PASS
9,3000.0,c2 min 12 max 10,0.379,3008.943,0.298,2.059,1196.544,1216.550,PASS


In [3]:
INK, GRID = "#52514e", "#e4e3df"
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"
# Frequency axes (values in MHz, log): labelled decades with the unit in the tick, minor grid in between.
FREQ_AXIS = dict(type="log", tickvals=[1, 10, 100, 1e3, 1e4, 1e5],
                 ticktext=["1 MHz", "10 MHz", "100 MHz", "1 GHz", "10 GHz", "100 GHz"],
                 minor=dict(showgrid=True, gridcolor="#f1f0ed"))


def freq_label(f_mhz):
    return f"{f_mhz / 1e3:g} GHz" if f_mhz >= 1e3 else f"{f_mhz:g} MHz"


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the check CSV, then show the figure."""
    base = Path(CHECK_CSV).with_name(f"{Path(CHECK_CSV).stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [4]:
# Per target: table curve (line), measured points (dots), predicted operating point (circle at f_target).
measured = summary[summary["kind"] == "summary"]
N_COLS = 4
n_rows = -(-len(measured) // N_COLS)
titles = [f"{freq_label(r.f_target / 1e6)}: c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} "
          f"max {r.vco_current_max:.0f} ({r.result})" for r in measured.itertuples()]
fig = make_subplots(rows=n_rows, cols=N_COLS, subplot_titles=titles, horizontal_spacing=0.06, vertical_spacing=0.09)
for i, r in enumerate(measured.itertuples()):
    row, col = i // N_COLS + 1, i % N_COLS + 1
    d = points[points["f_target"] == r.f_target].sort_values("vctrl_set")
    tab = d[d["kind"] == "table"]
    first = i == 0
    fig.add_scatter(x=tab["vctrl_set"], y=tab["f_table"] / 1e6, mode="lines", line=dict(color=BLUE, width=2),
                    name="lookup table", legendgroup="table", showlegend=first, row=row, col=col,
                    hovertemplate="table: Vctrl %{x:.3f} V, %{y:.1f} MHz<extra></extra>")
    fig.add_scatter(x=d["vctrl_set"], y=d["f_vco"] / 1e6, mode="markers", marker=dict(color=ORANGE, size=8),
                    name="measured", legendgroup="meas", showlegend=first, row=row, col=col,
                    customdata=np.c_[d["kind"], d["dev_pct"]],
                    hovertemplate="%{customdata[0]}: Vctrl %{x:.3f} V, %{y:.1f} MHz (%{customdata[1]:+.2f} %)"
                                  "<extra></extra>")
    fig.add_scatter(x=[r.vctrl_pred], y=[r.f_target / 1e6], mode="markers",
                    marker=dict(size=16, color="rgba(0,0,0,0)", line=dict(color=GREEN, width=2)),
                    name="target at predicted Vctrl", legendgroup="op", showlegend=first, row=row, col=col,
                    hovertemplate=f"target {r.f_target / 1e6:.1f} MHz at Vctrl {r.vctrl_pred:.3f} V, "
                                  f"measured {r.f_vco / 1e6:.1f} MHz<extra></extra>")
fig.update_layout(title="LUT check: measured vs table per target", template="plotly_white",
                  height=300 * n_rows, legend=dict(orientation="h", y=-0.06), hoverlabel=dict(namelength=-1))
fig.update_annotations(font_size=11)
fig.update_xaxes(gridcolor=GRID, range=[-0.03, 0.78])
fig.update_xaxes(title="Vctrl [V]", row=n_rows)
fig.update_yaxes(type="log", gridcolor=GRID)
fig.update_yaxes(title="f_vco [MHz]", col=1)
save(fig, "curves")
unreachable = summary[summary["kind"] == "unreachable"]["f_target"]
if len(unreachable):
    print("not reachable from the table:", ", ".join(freq_label(f / 1e6) for f in unreachable))

saved /users/micas/wvandest/projects/lagd-meas/results/vco/vco_lut_check_S5_20261002_102659_curves.html and .png


not reachable from the table: 50 MHz


In [5]:
# Measured vs table for every point, per target frequency; dashed = the test tolerance.
fig = go.Figure()
for b in (-F_TOL * 100, F_TOL * 100):
    fig.add_hline(y=b, line=dict(color=INK, width=1, dash="dash"))
fig.add_hline(y=0, line=dict(color=GRID, width=1))
for kind, color, label in (("table", BLUE, "table Vctrl points"), ("kvco", ORANGE, "Vctrl pred ± 25 mV"),
                           ("operating", GREEN, "predicted Vctrl")):
    d = points[points["kind"] == kind]
    fig.add_scatter(x=d["f_target"] / 1e6, y=d["dev_pct"], mode="markers", name=label,
                    marker=dict(color=color, size=9, line=dict(color="white", width=1)),
                    customdata=np.c_[d["vctrl_set"], d["f_vco"] / 1e6, d["f_table"] / 1e6],
                    hovertemplate=("target %{x:.0f} MHz, Vctrl %{customdata[0]:.3f} V<br>measured "
                                   "%{customdata[1]:.1f} MHz, table %{customdata[2]:.1f} MHz (%{y:+.2f} %)"
                                   "<extra></extra>"))
fig.update_layout(title="Measured vs lookup table, all points", template="plotly_white", height=500,
                  legend=dict(orientation="h", y=-0.18))
fig.update_xaxes(title="target f_vco", gridcolor=GRID, **FREQ_AXIS)
fig.update_yaxes(title="measured / table - 1 [%]", gridcolor=GRID, zeroline=False)
save(fig, "deviation")

saved /users/micas/wvandest/projects/lagd-meas/results/vco/vco_lut_check_S5_20261002_102659_deviation.html and .png


In [6]:
# Local |Kvco| at the operating point: measured (Vctrl pred ± 25 mV) vs predicted from the table; band = KVCO_TOL.
k = measured.dropna(subset=["kvco_meas"])
lo, hi = min(k["kvco_pred"].min(), k["kvco_meas"].min()) / 1.5e6, max(k["kvco_pred"].max(), k["kvco_meas"].max()) * 1.5 / 1e6
line = np.array([lo, hi])
fig = go.Figure()
fig.add_scatter(x=np.r_[line, line[::-1]], y=np.r_[line * (1 + KVCO_TOL), line[::-1] * (1 - KVCO_TOL)],
                fill="toself", fillcolor="rgba(42,120,214,0.10)", line_width=0, hoverinfo="skip",
                name=f"±{KVCO_TOL * 100:.0f} %")
fig.add_scatter(x=line, y=line, mode="lines", line=dict(color=INK, width=1, dash="dash"), hoverinfo="skip",
                name="measured = predicted")
fig.add_scatter(x=k["kvco_pred"] / 1e6, y=k["kvco_meas"] / 1e6, mode="markers+text",
                text=[freq_label(f / 1e6) for f in k["f_target"]], textposition="top center",
                textfont=dict(color=INK, size=10), marker=dict(color=BLUE, size=10, line=dict(color="white", width=1)),
                name="targets", hovertemplate="predicted %{x:.0f} MHz/V, measured %{y:.0f} MHz/V<extra>%{text}</extra>")
fig.update_layout(title="Local |Kvco| at the operating point", template="plotly_white", height=600, width=750,
                  legend=dict(orientation="h", y=-0.15))
fig.update_xaxes(title="|Kvco| predicted [MHz/V]", type="log", range=np.log10([lo, hi]), gridcolor=GRID)
fig.update_yaxes(title="|Kvco| measured [MHz/V]", type="log", range=np.log10([lo, hi]), gridcolor=GRID)
save(fig, "kvco")

saved /users/micas/wvandest/projects/lagd-meas/results/vco/vco_lut_check_S5_20261002_102659_kvco.html and .png
